# Aula 4, Structured output

Notebook executável que acompanha a aula [04-structured-output.md](../../lessons/modulo-08-prompt-engineering/04-structured-output.md).

## O que vamos fazer aqui

Para integrar um LLM a um sistema, precisamos de saída em formato fixo, como JSON. Vamos
construir um extrator e um validador de JSON, testá-los com respostas simuladas (inclusive
imperfeitas) e, no projeto, montar um tutor que explica em níveis e devolve respostas
estruturadas. As funções são Python puro e testáveis sem o modelo.

## Extrair e validar JSON

O extrator recupera o objeto JSON mesmo com texto em volta. O validador confere se os campos
esperados estão presentes. Testamos com respostas simuladas, incluindo casos com erro.

In [ ]:
import re
import json


def extrair_json(texto):
    """Recupera o primeiro objeto JSON do texto, mesmo com texto em volta."""
    m = re.search(r"\{.*\}", texto, re.DOTALL)   # do primeiro { ao último }
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def validar(dados, campos):
    """Confere se o JSON tem todos os campos esperados."""
    if dados is None:
        return False, "não é JSON válido"
    faltando = [c for c in campos if c not in dados]
    if faltando:
        return False, "faltam campos: " + ", ".join(faltando)
    return True, "ok"


respostas = [
    'Claro! {"tema": "derivada", "nivel": "basico", "explicacao": "..."}',
    '{"tema": "matriz"}',                       # falta nível e explicação
    "desculpe, não consegui responder",         # sem JSON
]
campos = ["tema", "nivel", "explicacao"]

for r in respostas:
    dados = extrair_json(r)
    ok, msg = validar(dados, campos)
    print(f"válido: {ok!s:5} | {msg}  ->  {dados}")

O extrator recupera o JSON mesmo precedido de texto; o validador aprova a resposta
completa, reprova a que tem campos faltando (apontando quais) e reprova a que não traz JSON.
Essa dupla permite usar a saída do modelo com segurança.

## Projeto do módulo, tutor em níveis com saída estruturada

Pedimos ao modelo uma explicação de um conceito em três níveis (iniciante, intermediário,
avançado), exigindo JSON com campos definidos, e processamos cada resposta com o extrator e o
validador. Roda com o Ollama; sem ele, a demonstração das funções acima já mostra o mecanismo.

In [ ]:
def prompt_tutor(conceito, nivel):
    return (
        "Você é um tutor. Responda APENAS com um objeto JSON, sem texto fora dele, "
        f'com os campos "nivel", "explicacao" e "exemplo".\n'
        f"Conceito: {conceito}. Nível do aluno: {nivel}."
    )


try:
    import ollama

    for nivel in ["iniciante", "intermediario", "avancado"]:
        r = ollama.chat(model="llama3.1",
                        messages=[{"role": "user", "content": prompt_tutor("derivada", nivel)}])
        dados = extrair_json(r["message"]["content"])
        ok, msg = validar(dados, ["nivel", "explicacao", "exemplo"])
        print(f"[{nivel}] válido: {ok} | {msg}")
        if dados:
            print("   explicação:", str(dados.get("explicacao"))[:120], "\n")
except Exception as erro:
    print("Ollama não disponível. As funções acima já demonstram a extração e validação.")
    print("Detalhe:", erro)

Para cada nível, o tutor devolve um JSON que extraímos e validamos, e a explicação
muda de profundidade conforme o nível. Para o projeto, compare as três explicações e avalie se
cada uma se ajusta ao público. Com isso, você fecha o prompt engineering e fica pronto para o
RAG do Módulo 9.